# 👗 Zintoo: AI-Powered Hyper-Local Fashion Intelligence Platform

**PS6 — End-to-End AI System**
- **Module 1:** Multimodal Fashion Recommendations (FashionCLIP + FAISS)
- **Module 2:** Hyper-Local Demand Forecasting (Prophet)
- **Module 3:** Agentic Inventory Orchestration (ReAct Agent)

> ⚡ **Run all cells sequentially. Enable GPU: Runtime → Change runtime type → GPU (T4)**

---

## 🔧 Cell 1 — Install Dependencies & Check GPU

In [ ]:
    # ── Install all required packages ──────────────────────────────
!pip install -q torch torchvision transformers faiss-cpu Pillow \
    pandas numpy scikit-learn prophet fastapi uvicorn python-multipart \
    streamlit plotly kaggle requests tqdm matplotlib

# ── Check GPU availability ─────────────────────────────────────
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cuda':
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f'✅ GPU detected: {gpu_name} ({gpu_mem:.1f} GB)')
else:
    print('⚠️  No GPU — running on CPU (embedding step will be slower)')
print(f'   PyTorch {torch.__version__}  •  Device: {device}')

## 🔧 Cell 2 — Configure Kaggle API

In [ ]:
import os, json
from pathlib import Path

kaggle_dir  = Path.home() / '.kaggle'
kaggle_json = kaggle_dir / 'kaggle.json'

if not kaggle_json.exists():
    try:
        from google.colab import files
        print('📁 Upload your kaggle.json file:')
        print('   (Download from https://www.kaggle.com/settings → API → Create New Token)')
        uploaded = files.upload()
        kaggle_dir.mkdir(parents=True, exist_ok=True)
        with open(kaggle_json, 'wb') as f:
            f.write(uploaded['kaggle.json'])
        os.chmod(str(kaggle_json), 0o600)
        print(f'✅ kaggle.json saved to {kaggle_json}')
    except ImportError:
        print('❌ Not in Colab. Place kaggle.json at ~/.kaggle/kaggle.json')
else:
    print(f'✅ kaggle.json already exists at {kaggle_json}')

## 📥 Cell 3 — Download & Extract Dataset

In [ ]:
import kaggle, zipfile

# ── Paths (relative to notebook CWD) ──────────────────────────
DATA_DIR    = Path('data')
DATASET_DIR = DATA_DIR / 'fashion-product-images-small'
IMAGES_DIR  = DATASET_DIR / 'images'
STYLES_CSV  = DATASET_DIR / 'styles.csv'

if not STYLES_CSV.exists():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    print('📥 Downloading dataset (~280 MB) …')
    kaggle.api.authenticate()
    kaggle.api.dataset_download_files(
        'paramaggarwal/fashion-product-images-small',
        path=str(DATA_DIR), unzip=False,
    )

    zp = DATA_DIR / 'fashion-product-images-small.zip'
    if zp.exists():
        print('📂 Extracting …')
        with zipfile.ZipFile(zp, 'r') as z:
            z.extractall(DATA_DIR)
        zp.unlink()

    # Handle nested extraction folder
    if not IMAGES_DIR.exists():
        for p in DATA_DIR.rglob('images'):
            if p.is_dir() and p.parent != DATASET_DIR:
                p.parent.rename(DATASET_DIR)
                break

import pandas as pd
df_raw = pd.read_csv(STYLES_CSV, on_bad_lines='skip')
n_imgs = len(list(IMAGES_DIR.glob('*.jpg')))
print(f'✅ Dataset ready — {len(df_raw):,} products  •  {n_imgs:,} images')

## 🧹 Cell 4 — Preprocess & Build Product Catalog

In [ ]:
import pandas as pd, numpy as np
from pathlib import Path

df = pd.read_csv(STYLES_CSV, on_bad_lines='skip')

# Keep relevant columns
cols = ['id','gender','masterCategory','subCategory','articleType',
        'baseColour','season','year','usage','productDisplayName']
df = df[[c for c in cols if c in df.columns]].copy()

# Drop rows missing critical fields
df.dropna(subset=['id','masterCategory','articleType','productDisplayName'], inplace=True)
df['id'] = df['id'].astype(int)

for c in ['gender','baseColour','season','usage']:
    if c in df.columns:
        df[c] = df[c].fillna('Unknown')

# Add image path & filter to images that exist on disk
df['image_path'] = df['id'].apply(lambda x: str(IMAGES_DIR / f'{x}.jpg'))
df['image_exists'] = df['image_path'].apply(lambda p: Path(p).exists())
df = df[df['image_exists']].drop(columns=['image_exists']).reset_index(drop=True)

# Build rich text description
def _desc(r):
    parts = [str(r['productDisplayName'])]
    if r.get('gender','Unknown') != 'Unknown': parts.append(f"for {r['gender']}")
    if r.get('baseColour','Unknown') != 'Unknown': parts.append(f"in {r['baseColour']}")
    if r.get('season','Unknown') != 'Unknown': parts.append(f"({r['season']} season)")
    if r.get('usage','Unknown') != 'Unknown': parts.append(f"- {r['usage']} wear")
    return ' '.join(parts)

df['description'] = df.apply(_desc, axis=1)
df['category_path'] = (df['masterCategory'].astype(str) + ' > '
                       + df['subCategory'].astype(str) + ' > '
                       + df['articleType'].astype(str))

catalog = df.copy()
catalog.to_csv(DATASET_DIR / 'catalog.csv', index=False)

print(f'✅ Catalog built — {len(catalog):,} products')
print(catalog['masterCategory'].value_counts().head())

---
## 🤖 MODULE 1 — Multimodal Recommendations (FashionCLIP + FAISS)
---

### Cell 5 — Extract FashionCLIP Embeddings & Build FAISS Index

In [ ]:
import torch, faiss, pickle, numpy as np
from PIL import Image
from tqdm.auto import tqdm
from transformers import CLIPModel, CLIPProcessor

EMBEDDINGS_DIR  = Path('outputs/embeddings')
EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)
FAISS_INDEX_PATH = EMBEDDINGS_DIR / 'fashion_index.faiss'
PRODUCT_MAP_PATH = EMBEDDINGS_DIR / 'product_map.pkl'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

MODEL_NAME = 'patrickjohncyh/fashion-clip'
print(f'Loading {MODEL_NAME} …')
model     = CLIPModel.from_pretrained(MODEL_NAME).to(device)
processor = CLIPProcessor.from_pretrained(MODEL_NAME)
model.eval()
print('✅ FashionCLIP loaded')

# ── Encode all catalog images ──────────────────────────────────
BATCH    = 64
paths    = catalog['image_path'].tolist()
all_emb  = []

for i in tqdm(range(0, len(paths), BATCH), desc='Encoding images'):
    batch = paths[i:i+BATCH]
    imgs  = []
    for p in batch:
        try:
            imgs.append(Image.open(p).convert('RGB'))
        except Exception:
            imgs.append(Image.new('RGB', (224, 224), (128, 128, 128)))

    inputs = processor(images=imgs, return_tensors='pt', padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        out = model.get_image_features(**inputs)
    emb = out / out.norm(dim=-1, keepdim=True)
    all_emb.append(emb.cpu().numpy())

embeddings = np.vstack(all_emb).astype('float32')
print(f'Embeddings shape: {embeddings.shape}')

# ── Build & save FAISS index ───────────────────────────────────
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
faiss.write_index(index, str(FAISS_INDEX_PATH))

with open(PRODUCT_MAP_PATH, 'wb') as f:
    pickle.dump({
        'product_ids': catalog['id'].tolist(),
        'catalog': catalog.to_dict('records'),
    }, f)

np.save(EMBEDDINGS_DIR / 'embeddings.npy', embeddings)
print(f'✅ FAISS index saved — {index.ntotal} vectors')

### Cell 6 — Recommendation Engine + Text & Image Demo

In [ ]:
# ── Helper: encode text queries ────────────────────────────────
def encode_text(texts):
    inputs = processor(text=texts, return_tensors='pt', padding=True, truncation=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        out = model.get_text_features(**inputs)
    return (out / out.norm(dim=-1, keepdim=True)).cpu().numpy().astype('float32')


def recommend_text(query, k=5):
    """Return top-k products by text similarity."""
    emb = encode_text([query])
    scores, idxs = index.search(emb, k)
    results = []
    for idx, score in zip(idxs[0], scores[0]):
        p = catalog.iloc[idx]
        results.append({
            'rank': len(results)+1, 'score': round(float(score), 4),
            'name': p['productDisplayName'],
            'category': p['masterCategory'],
            'type': p['articleType'],
            'color': p.get('baseColour', ''),
            'image': p['image_path'],
        })
    return results


def recommend_image(img_path, k=5):
    """Return top-k products by image similarity."""
    img = Image.open(img_path).convert('RGB')
    inputs_ = processor(images=[img], return_tensors='pt', padding=True)
    inputs_ = {k_: v.to(device) for k_, v in inputs_.items()}
    with torch.no_grad():
        out = model.get_image_features(**inputs_)
    emb = (out / out.norm(dim=-1, keepdim=True)).cpu().numpy().astype('float32')
    scores, idxs = index.search(emb, k+1)
    results = []
    for idx, score in zip(idxs[0], scores[0]):
        if score > 0.999:      # skip exact self-match
            continue
        p = catalog.iloc[idx]
        results.append({
            'rank': len(results)+1, 'score': round(float(score), 4),
            'name': p['productDisplayName'],
            'category': p['masterCategory'],
            'type': p['articleType'],
            'color': p.get('baseColour', ''),
            'image': p['image_path'],
        })
        if len(results) >= k:
            break
    return results


# ── Demo ────────────────────────────────────────────────────────
print('=' * 60)
print('🎯 RECOMMENDATION DEMO — Text Queries')
print('=' * 60)

demo_queries = [
    'casual kurta for a college fest',
    'formal black shoes for office',
    'summer floral dress for women',
]
for q in demo_queries:
    print(f"\n📝 Query: '{q}'")
    for r in recommend_text(q, 5):
        print(f"   {r['rank']}. [{r['score']:.4f}] {r['name']} "
              f"({r['category']} > {r['type']})")

print(f"\n🖼️ Image Query — using first catalog product")
src = catalog.iloc[0]
print(f"   Source: {src['productDisplayName']}")
for r in recommend_image(src['image_path'], 5):
    print(f"   {r['rank']}. [{r['score']:.4f}] {r['name']}")

### Cell 7 — Visual Recommendation Results

In [ ]:
import matplotlib.pyplot as plt

def show_recommendations(query, k=5):
    results = recommend_text(query, k)
    fig, axes = plt.subplots(1, k, figsize=(20, 4))
    fig.suptitle(f'Recommendations for: "{query}"', fontsize=14, fontweight='bold')
    for ax, r in zip(axes, results):
        try:
            img = Image.open(r['image'])
            ax.imshow(img)
        except Exception:
            pass
        ax.set_title(f"#{r['rank']} ({r['score']:.3f})\n{r['name'][:30]}", fontsize=8)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

show_recommendations('casual kurta for a college fest')
show_recommendations('sporty running shoes')
show_recommendations('ethnic wear for wedding')

---
## 🔧 SYNTHETIC DATA GENERATION — Warehouses, Weather & Demand
---

### Cell 8 — Generate Synthetic Data

In [ ]:
import random, requests
from datetime import datetime, timedelta

# ── Configuration ──────────────────────────────────────────────
PIN_CODES     = ['400001', '400002', '400003', '400004', '400005']
WAREHOUSE_IDS = ['W1', 'W2', 'W3', 'W4', 'W5']
WH_PIN        = dict(zip(WAREHOUSE_IDS, PIN_CODES))
HISTORY_DAYS  = 90

# Pick 20 random SKUs from catalog
skus = catalog.sample(20, random_state=42)['id'].astype(str).tolist()

# ── 1. Warehouse Inventory ─────────────────────────────────────
random.seed(42)
inv_rows = []
for sku in skus:
    for wh in WAREHOUSE_IDS:
        inv_rows.append({
            'product_id': sku,
            'sku': f'SKU-{sku}',
            'warehouse_id': wh,
            'pincode': WH_PIN[wh],
            'current_stock': random.randint(5, 100),
            'reorder_threshold': 10,
            'max_capacity': 100,
            'last_restocked': (datetime.now() - timedelta(days=random.randint(1,14))).strftime('%Y-%m-%d'),
        })
inventory_df = pd.DataFrame(inv_rows)
inventory_df.to_csv(DATA_DIR / 'warehouse_inventory.csv', index=False)
print(f'✅ Warehouse inventory: {len(inventory_df)} rows')

# ── 2. Synthetic Weather Data ──────────────────────────────────
np.random.seed(42)
ts = pd.date_range(end=datetime.now(), periods=HISTORY_DAYS*24, freq='h')
temp   = 30 + 5*np.sin(2*np.pi*(ts.hour - 6)/24) + np.random.normal(0, 2, len(ts))
precip = np.zeros(len(ts))
for rd in np.random.choice(range(0, len(ts), 24), size=HISTORY_DAYS//5, replace=False):
    for h in range(np.random.randint(2, 8)):
        if rd+h < len(ts):
            precip[rd+h] = np.random.exponential(5)
weather_df = pd.DataFrame({
    'timestamp': ts,
    'temperature': temp,
    'precipitation': precip,
    'weathercode': np.where(precip > 0, 61, 0),
})
weather_df.to_csv(DATA_DIR / 'weather_data.csv', index=False)
print(f'✅ Weather data: {len(weather_df)} rows')

# ── 3. Demand History ──────────────────────────────────────────
np.random.seed(42); random.seed(42)
demand_rows = []
for sku in skus:
    base = np.random.uniform(2, 8)
    for pin in PIN_CODES:
        pin_mult = np.random.uniform(0.5, 1.5)
        for i, t in enumerate(ts):
            h_eff = (0.3 + 0.7*np.exp(-0.5*((t.hour-10)/3)**2)
                         + 0.5*np.exp(-0.5*((t.hour-18)/2)**2))
            if t.hour < 6 or t.hour > 23:
                h_eff *= 0.1
            dow_eff = 1.4 if t.dayofweek >= 5 else 1.0
            w_eff   = 0.6 if precip[i] > 5 else (0.8 if precip[i] > 0 else 1.0)
            ev      = hash(f'{sku}_{pin}_{t.date()}') % 100
            ev_eff  = 2.5 if ev < 5 else (3.0 if ev < 10 else 1.0)
            d = max(0, int(np.random.poisson(max(0.1, base*pin_mult*h_eff*dow_eff*w_eff*ev_eff))))
            demand_rows.append({
                'timestamp': t, 'sku': f'SKU-{sku}', 'pincode': pin,
                'demand': d, 'net_demand': d,
                'is_weekend': 1 if t.dayofweek >= 5 else 0,
                'hour': t.hour, 'day_of_week': t.strftime('%A'),
                'returns': 0,
            })

demand_df = pd.DataFrame(demand_rows)
demand_df.to_csv(DATA_DIR / 'demand_history.csv', index=False)
print(f'✅ Demand history: {len(demand_df):,} rows')

---
## 📈 MODULE 2 — Hyper-Local Demand Forecasting (Prophet)
---

### Cell 9 — Train Prophet Forecast + Interactive Plot

In [ ]:
import warnings; warnings.filterwarnings('ignore')
from prophet import Prophet
import plotly.graph_objects as go

FORECASTS_DIR = Path('outputs/forecasts')
FORECASTS_DIR.mkdir(parents=True, exist_ok=True)

demand_data = pd.read_csv(DATA_DIR / 'demand_history.csv', parse_dates=['timestamp'])
weather     = pd.read_csv(DATA_DIR / 'weather_data.csv',   parse_dates=['timestamp'])

# Pick the first SKU-pincode pair
test_sku = demand_data['sku'].unique()[0]
test_pin = demand_data['pincode'].unique()[0]

mask   = (demand_data['sku'] == test_sku) & (demand_data['pincode'] == test_pin)
subset = demand_data[mask].copy()

# Prepare Prophet dataframe
df_p = pd.DataFrame({
    'ds': subset['timestamp'],
    'y':  subset['net_demand'],
    'is_weekend': subset['is_weekend'].astype(float),
})
df_p = df_p.merge(
    weather[['timestamp','temperature','precipitation']].rename(columns={'timestamp':'ds'}),
    on='ds', how='left',
)
df_p['temperature'].fillna(30, inplace=True)
df_p['precipitation'].fillna(0, inplace=True)
df_p.sort_values('ds', inplace=True)

# Train / Test split: last 24 h held out
split = df_p['ds'].max() - timedelta(hours=24)
train = df_p[df_p['ds'] <= split]
test  = df_p[df_p['ds'] >  split]

# Fit Prophet
m = Prophet(
    daily_seasonality=True, weekly_seasonality=True, yearly_seasonality=False,
    changepoint_prior_scale=0.05, seasonality_mode='multiplicative',
)
m.add_regressor('is_weekend',    mode='multiplicative')
m.add_regressor('temperature')
m.add_regressor('precipitation', mode='multiplicative')
m.fit(train)

# Predict
future = m.make_future_dataframe(periods=24, freq='h')
future = future.merge(df_p[['ds','is_weekend','temperature','precipitation']], on='ds', how='left')
future['is_weekend'].fillna(future['ds'].dt.dayofweek.apply(lambda x: 1.0 if x >= 5 else 0.0), inplace=True)
future['temperature'].fillna(30, inplace=True)
future['precipitation'].fillna(0, inplace=True)

fc = m.predict(future)
fc['yhat']       = fc['yhat'].clip(lower=0)
fc['yhat_lower'] = fc['yhat_lower'].clip(lower=0)

# ── Metrics on test set ────────────────────────────────────────
tf = fc[fc['ds'].isin(test['ds'])].merge(test[['ds','y']], on='ds')
if len(tf) > 0:
    act, pred = tf['y'].values, tf['yhat'].values
    rmse_val = float(np.sqrt(np.mean((act - pred)**2)))
    nz = act > 0
    mape_val = float(np.mean(np.abs((act[nz] - pred[nz]) / act[nz])) * 100) if nz.sum() > 0 else 0
    print(f'📊 Forecast Metrics — RMSE: {rmse_val:.2f}  •  MAPE: {mape_val:.1f}%')

# ── Plotly chart ────────────────────────────────────────────────
hist      = df_p.tail(168)        # last 7 days
fc_future = fc[fc['ds'] > split]  # forecast window

fig = go.Figure()
fig.add_trace(go.Scatter(x=hist['ds'], y=hist['y'],
                         name='Historical', line=dict(color='#3498db')))
fig.add_trace(go.Scatter(x=fc_future['ds'], y=fc_future['yhat'],
                         name='Forecast',   line=dict(color='#e74c3c', dash='dash')))
fig.add_trace(go.Scatter(
    x=pd.concat([fc_future['ds'], fc_future['ds'][::-1]]),
    y=pd.concat([fc_future['yhat_upper'], fc_future['yhat_lower'][::-1]]),
    fill='toself', fillcolor='rgba(231,76,60,0.15)',
    line=dict(color='rgba(0,0,0,0)'), name='95% CI',
))
fig.update_layout(
    title=f'Demand Forecast: {test_sku} @ {test_pin}',
    template='plotly_dark', height=500,
)
fig.show()

---
## 🤖 MODULE 3 — Agentic Inventory Orchestration (ReAct Agent)
---

### Cell 10 — Run Autonomous Inventory Rebalancing

In [ ]:
inventory = pd.read_csv(DATA_DIR / 'warehouse_inventory.csv')

# ── Lightweight agent toolkit ──────────────────────────────────
class SimpleToolkit:
    def __init__(self, inv):
        self.inventory = inv.copy()
        self.log = []

    def check_stock(self, wh, sku):
        m = (self.inventory['warehouse_id'] == wh) & (self.inventory['sku'] == sku)
        if m.sum() == 0:
            return {'warehouse_id': wh, 'sku': sku, 'current_stock': 0,
                    'status': 'not_found', 'reorder_threshold': 10,
                    'max_capacity': 100, 'needs_restock': True, 'surplus': 0, 'deficit': 10}
        r = self.inventory[m].iloc[0]
        s, t = int(r['current_stock']), int(r['reorder_threshold'])
        return {
            'warehouse_id': wh, 'sku': sku,
            'pincode': r.get('pincode', ''),
            'current_stock': s, 'reorder_threshold': t,
            'max_capacity': int(r.get('max_capacity', 100)),
            'status': 'critical' if s <= t//2 else ('low' if s <= t else 'healthy'),
            'needs_restock': s <= t,
            'surplus': max(0, s - t*2),
            'deficit': max(0, t - s),
        }

    def transfer(self, fr, to, sku, qty):
        src = self.check_stock(fr, sku)
        dst = self.check_stock(to, sku)
        if src['current_stock'] < qty:
            return {'success': False, 'error': 'Insufficient stock'}
        if dst['current_stock'] + qty > dst['max_capacity']:
            return {'success': False, 'error': 'Exceeds capacity'}
        self.inventory.loc[
            (self.inventory['warehouse_id'] == fr) & (self.inventory['sku'] == sku),
            'current_stock'] -= qty
        self.inventory.loc[
            (self.inventory['warehouse_id'] == to) & (self.inventory['sku'] == sku),
            'current_stock'] += qty
        rec = {'from': fr, 'to': to, 'sku': sku, 'qty': qty, 'success': True}
        self.log.append(rec)
        return rec


tk = SimpleToolkit(inventory)

print('=' * 60)
print('🤖 AGENTIC ORCHESTRATION — Inventory Rebalancing')
print('=' * 60)

# ── Phase 1: OBSERVE ───────────────────────────────────────────
print('\n[1] 🔍 OBSERVE: Scanning inventory …')
critical, surplus = [], []
for sku in inventory['sku'].unique():
    for wh in WAREHOUSE_IDS:
        info = tk.check_stock(wh, sku)
        if info['status'] == 'critical':
            critical.append(info)
        elif info.get('surplus', 0) > 0:
            surplus.append(info)
print(f'   Critical: {len(critical)}  •  Surplus: {len(surplus)}')

# ── Phase 2: THINK ─────────────────────────────────────────────
print('\n[2] 🧠 THINK: Planning transfers …')
surplus_map = {}
for s in surplus:
    surplus_map.setdefault(s['sku'], []).append(s)

plans = []
for need in critical:
    sources = surplus_map.get(need['sku'], [])
    if not sources:
        continue
    src = max(sources, key=lambda x: x['surplus'])
    qty = min(need['deficit'] + 10, src['surplus'], 30)
    if qty > 0:
        plans.append({
            'sku': need['sku'], 'from': src['warehouse_id'],
            'to': need['warehouse_id'], 'qty': qty,
            'reason': (f"Forecasted demand spike. Stock at {need['warehouse_id']}: "
                       f"{need['current_stock']} (threshold: {need['reorder_threshold']}). "
                       f"Source {src['warehouse_id']} has surplus."),
        })
print(f'   Planned {len(plans)} transfers')

# ── Phase 3: ACT ───────────────────────────────────────────────
print('\n[3] ⚡ ACT: Executing transfers …')
for p in plans:
    r = tk.transfer(p['from'], p['to'], p['sku'], p['qty'])
    e = '✅' if r.get('success') else '❌'
    print(f"   {e} Transfer {p['qty']}× {p['sku']}: {p['from']} → {p['to']}")
    print(f"      Reason: {p['reason'][:80]}…")

# ── Phase 4: REFLECT ───────────────────────────────────────────
print('\n[4] 📊 REFLECT: Impact assessment')
ok = [l for l in tk.log if l.get('success')]
print(f'   Successful transfers: {len(ok)}')
print(f'   Total units moved:    {sum(l["qty"] for l in ok)}')
print(f'   Est. cost:            ₹{sum(l["qty"] * 5 for l in ok)}')
print('\n📋 Transfer Log:')
for l in tk.log[:10]:
    print(f'   {l}')

---
## 📊 MODULE 4 — Evaluation Report
---

### Cell 11 — Full Evaluation (Precision@K, NDCG@K, MAPE, RMSE, SLA)

In [ ]:
print('=' * 60)
print('📊 FULL EVALUATION REPORT')
print('=' * 60)

# ── 1. Recommendation Quality ──────────────────────────────────
print('\n--- Recommendation Quality ---')
np.random.seed(42)
sample_idxs = np.random.choice(len(catalog), size=50, replace=False)

for K in [5, 10, 20]:
    precs, ndcgs = [], []
    for idx in sample_idxs:
        p   = catalog.iloc[idx]
        cat = p['masterCategory']
        recs     = recommend_text(p['productDisplayName'], K)
        rec_cats = [r['category'] for r in recs]

        prec = sum(1 for c in rec_cats if c == cat) / K
        dcg  = sum((1.0 if c == cat else 0) / np.log2(i+2) for i, c in enumerate(rec_cats))
        ideal = sum(1/np.log2(i+2) for i in range(min(sum(1 for c in rec_cats if c == cat), K)))
        ndcg = dcg / ideal if ideal > 0 else 0

        precs.append(prec)
        ndcgs.append(ndcg)

    print(f'   Precision@{K}: {np.mean(precs):.4f}   NDCG@{K}: {np.mean(ndcgs):.4f}')

# ── 2. Forecast Accuracy ───────────────────────────────────────
print(f'\n--- Forecast Accuracy ---')
print(f'   MAPE: {mape_val:.1f}%')
print(f'   RMSE: {rmse_val:.2f}')

# ── 3. SLA Fulfillment Rate ────────────────────────────────────
print(f'\n--- SLA Fulfillment ---')
np.random.seed(42)
fulfilled = sum(
    1 for _ in range(100)
    if tk.check_stock(
        np.random.choice(WAREHOUSE_IDS),
        np.random.choice(inventory['sku'].unique()),
    )['current_stock'] > 0
)
print(f'   SLA Fulfillment Rate: {fulfilled}%')

print('\n' + '=' * 60)
print('✅ ALL EVALUATIONS COMPLETE')
print('=' * 60)

---
## 🌐 OPTIONAL — Run FastAPI Backend (in Colab)

In [ ]:
# This cell prints instructions for running the FastAPI & Streamlit servers.
# They are designed for local use but can be tunnelled via ngrok in Colab.

print('To run the FastAPI backend locally:')
print('  cd zintoo && uvicorn api.main:app --host 0.0.0.0 --port 8000 --reload')
print()
print('To run the Streamlit dashboard:')
print('  cd zintoo && streamlit run dashboard/app.py')
print()
print('API Endpoints:')
print('  POST /recommend         — Text-based recommendations')
print('  POST /recommend/image   — Image / multimodal recommendations')
print('  GET  /forecast/{sku}/{pincode} — Demand forecast')
print('  POST /orchestrate       — Run inventory rebalancing')
print('  GET  /health            — Health check')

---
## 🐛 Debugging Guide

In [ ]:
debugging_guide = '''
╔══════════════════════════════════════════════════════════╗
║  🐛 DEBUGGING AGENT — Common Issues & Fixes             ║
╚══════════════════════════════════════════════════════════╝

1. KAGGLE API ERRORS
   ❌ "Could not find kaggle.json"
   ✅ Upload kaggle.json in Cell 2, or place at ~/.kaggle/kaggle.json
   ✅ chmod 600 ~/.kaggle/kaggle.json

   ❌ "403 Forbidden" when downloading
   ✅ Accept dataset rules on Kaggle website first
   ✅ Check API key is valid (regenerate if needed)

2. FILE PATH ISSUES
   ❌ "FileNotFoundError: styles.csv"
   ✅ Check DATASET_DIR points to correct location
   ✅ Verify zip extraction: !ls data/fashion-product-images-small/

   ❌ Images not found
   ✅ !ls data/fashion-product-images-small/images/ | head

3. MODEL NOT LOADING
   ❌ "CUDA out of memory"
   ✅ Reduce BATCH to 16 or 32 in Cell 5
   ✅ Or force CPU: device = "cpu"

   ❌ "Connection error" downloading model
   ✅ Model is ~600 MB, wait for download
   ✅ Or pre-download: !huggingface-cli download patrickjohncyh/fashion-clip

4. EMPTY / IRRELEVANT RECOMMENDATIONS
   ❌ Empty results
   ✅ Verify FAISS index: print(index.ntotal)
   ✅ Check embeddings shape matches catalog length

   ❌ Irrelevant results
   ✅ FashionCLIP excels at fashion-specific queries
   ✅ Use specific queries: "blue cotton kurta" not just "clothing"

5. PROPHET ERRORS
   ❌ "No module named prophet"
   ✅ !pip install prophet

   ❌ Forecast all zeros
   ✅ Check demand data is not all zeros
   ✅ Verify timestamp column is properly parsed as datetime

6. STREAMLIT / API ISSUES
   ❌ "Address already in use"
   ✅ Kill existing: !kill $(lsof -t -i:8501)
   ✅ Or use different port: streamlit run app.py --server.port 8502
'''
print(debugging_guide)